# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My lane is a **scoring** task, with a regression model underneath. The model predicts the expected CTR for a page from its features (position, content type, intent, length and demand). The score is how far the page's actual CTR falls below that expectation, and pages are ranked by that score.

I chose scoring over classification because there is no ready-made "needs review" label in the data, and a ranked list lets a content team work from the top down within the time they have. I chose it over clustering because my question is about a gap from what is expected, not about grouping similar pages.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**What I would predict:** a page's 90-day CTR (clicks divided by impressions), using features that are known before clicks happen: average position, content type, main intent, word count, search volume and competition. The prediction is the "expected CTR" for that kind of page.

**Where the label comes from:** CTR is an observed outcome measured over 90 days, not a rule I made up. The review score is then a defined rule: expected CTR minus actual CTR. Pages furthest below expectation rank first.

**Leakage guard:** columns that are results of clicks (clicks, sessions, pageviews, users) stay out of the features, otherwise the model would just copy the answer. Pages with very few impressions are excluded because their CTR is mostly noise.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Metric:** mean absolute error (MAE) of predicted CTR on held-out pages the model has not seen, compared with a simple baseline that predicts each page's position-tier average CTR.

**What "good" means:** the model's MAE is clearly lower than the baseline's. As a starting target I will aim for at least 10% lower. If the model cannot beat the baseline, the simple tier rule is the better tool and I will say so honestly. I will adjust the target once I see the baseline MAE from my data (printed in section 4).

**Why this metric:** MAE is in the same units as CTR, so a content team can read it, and the baseline comparison keeps the claim modest: it only says the model does better than a simple rule.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)

MIN_IMPRESSIONS = 100   # provisional cut-off so CTR is not just noise
slice_df = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= MIN_IMPRESSIONS)].copy()

slice_df["position_tier"] = pd.cut(slice_df["avg_position"], [0, 3, 10, 20, 10000],
                                   labels=["1-3", "4-10", "11-20", "20+"])
slice_df["tier_baseline_ctr"] = slice_df.groupby("position_tier", observed=True)["ctr"].transform("mean")
baseline_mae = (slice_df["ctr"] - slice_df["tier_baseline_ctr"]).abs().mean()

show = [c for c in ["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
                    "position_tier", "content_type", "main_intent", "word_count"]
        if c in slice_df.columns]

print("Pages in the full file:", len(df))
print("Pages in my slice (one row = one page):", len(slice_df))
print("Baseline MAE (tier-average CTR):", round(baseline_mae, 4))
display(slice_df[show].head(10))

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
Pages in the full file: 30000
Pages in my slice (one row = one page): 22006
Baseline MAE (tier-average CTR): 0.2268


,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,content_type,main_intent,word_count
0,content_304f48230142,3803,29,0.76,10.6,11-20,keyword article,transactional,3221.0
1,content_a1fb4e703a9e,15320,7,0.05,20.3,20+,keyword article,informational,2481.0
2,content_9aa793d4d895,12581,11,0.09,36.5,20+,keyword article,informational,3515.0
3,content_331d6c4de07b,11751,58,0.49,6.2,4-10,keyword article,commercial,NaN
4,content_d99b7a2d90ca,19140,24,0.13,44.0,20+,keyword article,informational,2803.0
5,content_d4084a4bc775,3970,1,0.03,8.5,4-10,keyword article,transactional,3080.0
7,content_a63219c6e95a,1724,1,0.06,21.2,20+,keyword article,commercial,NaN
8,content_5e6c160719bc,32574,29,0.09,46.0,20+,keyword article,informational,3807.0
9,content_c27558df2b0c,1240,2,0.16,4.9,4-10,keyword article,informational,NaN
10,content_d8ee6cc6d642,20919,324,1.55,2.2,1-3,keyword article,commercial,NaN


**One row = one page**, identified by an anonymized content ID, with its 90-day impressions, clicks, CTR, average position and page features. My slice keeps pages that have an average position and at least 100 impressions in 90 days, so CTR is not just noise. The cut-off is provisional and I may tune it later. The baseline MAE shown is the error of simply predicting each tier's average CTR, which is the number my model needs to beat.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


A fixed rule such as "flag pages with CTR below 1.0" ignores position. In my starter data, average CTR drops from about 2.71 in positions 1-3 to 0.65 in positions 4-10, 0.32 in positions 11-20 and 0.21 beyond position 20. A single cutoff would flag the average page in every tier below the top, while missing pages in the top tier that underperform for their position.

Expected CTR also depends on several things at once: position, content type, intent and demand. These interact, and there are too many combinations to write as if-statements. A model can estimate expected CTR from all of them together, and a tier-average rule stays as the baseline it has to beat.

It is still a directional, observational tool. It shows which pages fall below what is expected, not why.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.